In [12]:
import pandas as pd
from scipy.stats import chi2_contingency
from itertools import combinations
from statsmodels.stats.multitest import multipletests

In [13]:
# import after saving these in notebook 07

root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/" # MAC
#root = "Y:/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/" # PC

df_augmin = pd.read_csv(root + "Cohort_for_stats_Augmin.csv")
df_USP28 = pd.read_csv(root + "Cohort_for_stats_USP28.csv")
df_ctrl = pd.read_csv(root + "Cohort_for_stats_ctrl.csv")

Chi-square test of independence

Tests whether cell category distribution depends on time bin.

Null hypothesis: distribution of 0/1/2 is the same in all bins.

Alternative: distributions differ.

Used widely for contingency tables.

In [14]:
def getstatistics(counts):
    # Because your data are counts in a contingency table, the most statistically appropriate and stable method for pairwise bin comparisons is actually:
    # Chi-square test on the 3×2 table (categories × bins)
    # This avoids the multinomial regression instability entirely.

    table = counts.pivot_table(
    index = "category",
    columns = "time_bin",
    values = "count",
    aggfunc = "sum",
    fill_value = 0
    )
    print("Global results")
    print(chi2_contingency(table))
    
    
    expanded = counts.loc[counts.index.repeat(counts["count"])].drop(columns = "count")


    
    def compare_bins(counts_df, bin1, bin2):

        subset = counts_df[counts_df["time_bin"].isin([bin1, bin2])]
    
        table = subset.pivot_table(
            index="category",
            columns="time_bin",
            values="count",
            aggfunc="sum",
            fill_value=0
        )

        chi2, p, dof, expected = chi2_contingency(table)

        return p

    bins = counts["time_bin"].unique()

    results = []

    for b1, b2 in combinations(bins, 2):
        p = compare_bins(counts, b1, b2)
        results.append((b1, b2, p))
    
    pairwise = pd.DataFrame(results, columns=["bin1","bin2","p"])
    
    
    
    pairwise["p_adj"] = multipletests(pairwise["p"], method="fdr_bh")[1] # this is the Multiple testing correction (Benjamini–Hochberg (FDR))

    print("Pairwise results")
    print(pairwise)    

In [15]:
getstatistics(df_augmin)

Global results
Chi2ContingencyResult(statistic=25.62511786850367, pvalue=0.00026143578773210125, dof=6, expected_freq=array([[ 3.39869281, 11.89542484,  8.32679739,  2.37908497],
       [ 4.18300654, 14.64052288, 10.24836601,  2.92810458],
       [12.41830065, 43.46405229, 30.4248366 ,  8.69281046]]))
Pairwise results
      bin1       bin2         p     p_adj
0  030-060    060-090  0.243949  0.243949
1  030-060    090-120  0.062156  0.090731
2  030-060  > 120 min  0.000532  0.001596
3  060-090    090-120  0.075609  0.090731
4  060-090  > 120 min  0.000366  0.001596
5  090-120  > 120 min  0.020460  0.040920


In [16]:
getstatistics(df_USP28)

Global results
Chi2ContingencyResult(statistic=9.830314952329765, pvalue=0.13198211259512072, dof=6, expected_freq=array([[ 1.16037736,  2.43396226,  1.83962264,  0.56603774],
       [ 7.54245283, 15.82075472, 11.95754717,  3.67924528],
       [32.29716981, 67.74528302, 51.20283019, 15.75471698]]))
Pairwise results
      bin1       bin2         p     p_adj
0  030-060    060-090  0.948671  0.948671
1  030-060    090-120  0.158329  0.295941
2  030-060  > 120 min  0.197294  0.295941
3  060-090    090-120  0.042178  0.253068
4  060-090  > 120 min  0.092360  0.277080
5  090-120  > 120 min  0.724142  0.868971


In [17]:
getstatistics(df_ctrl)

Global results
Chi2ContingencyResult(statistic=9.353037905669485, pvalue=0.1546752197565388, dof=6, expected_freq=array([[ 6.22641509,  4.18867925,  1.47169811,  0.11320755],
       [19.71698113, 13.26415094,  4.66037736,  0.35849057],
       [29.05660377, 19.54716981,  6.86792453,  0.52830189]]))
Pairwise results
      bin1       bin2         p    p_adj
0  030-060    060-090  0.313165  0.45525
1  030-060    090-120  0.025882  0.15529
2  030-060  > 120 min  0.481700  0.48170
3  060-090    090-120  0.330541  0.45525
4  060-090  > 120 min  0.328699  0.45525
5  090-120  > 120 min  0.379375  0.45525


In [18]:
df_cohort_stats = pd.read_csv(root + "Cohort_for_stats.csv")
df_cohort_stats.Condition.unique()

array(['3_siLUC1siHAUS6', '1_siLUC1', '4_siUSP28siHAUS6'], dtype=object)

In [20]:
# ANOVA Testing
# ANOVA as generalized linear model (GLM):
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols
import pingouin as pg
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multicomp import MultiComparison

measurement = 'Mitotic_duration_mins'
group_variable = 'Condition'
nan_elements = df_cohort_stats[measurement].isnull()
data = df_cohort_stats[~nan_elements]

# Bartlett's test for equal variances (One-way ANOVA requires equal variances!)

BartlettResult = stats.bartlett(
    data[data.Condition == '1_siLUC1'][measurement], 
    data[data.Condition == '3_siLUC1siHAUS6'][measurement], 
    data[data.Condition == '4_siUSP28siHAUS6'][measurement]
)

print("The Bartlett test for equal variances of {}: ".format(measurement) + str(BartlettResult))


#results = ols('Aspect-Ratio~ C('+group_variable+')', data=data).fit()
results = ols(measurement + '~ C('+group_variable+')', data = data).fit()
print(results.summary())

aov_table = sm.stats.anova_lm(results, typ = 2)

def anova_table(aov):
    aov['mean_sq'] = aov[:]['sum_sq'] / aov[:]['df']
    
    aov['eta_sq'] = aov[:-1]['sum_sq'] / sum(aov['sum_sq'])
    
    aov['omega_sq'] = (aov[:-1]['sum_sq']-(aov[:-1]['df'] * aov['mean_sq'][-1])) / (sum(aov['sum_sq']) + aov['mean_sq'][-1])
    
    cols = ['sum_sq', 'df', 'mean_sq', 'F', 'PR(>F)', 'eta_sq', 'omega_sq']
    aov = aov[cols]
    return aov

aov_table = anova_table(aov_table)
print("\n ANOVA TABLE: ")
print(aov_table)

# Post-hoc testing
mc = MultiComparison(data[measurement], data[group_variable])
mc_results = mc.tukeyhsd()
print("\n\n POST-HOC testing for {}: \n".format(measurement))
print(mc_results)
print("If \"reject\" = True, then H0 should be rejected")

# Welch's ANOVA when variances are unequal
aov_table_WELCH = pg.welch_anova(dv = measurement, between = group_variable, data = data)
print("\n Welch's ANOVA table") 
print(aov_table_WELCH)

# Post-hoc testing using Games-Howell post-hoc test
mc_results_GamesHowell = pg.pairwise_gameshowell(dv = measurement, between = group_variable, data = data)
print("\n Games-Howell post-hoc test table") 
print(mc_results_GamesHowell)

The Bartlett test for equal variances of Mitotic_duration_mins: BartlettResult(statistic=22.023215875670477, pvalue=1.6508949373302596e-05)
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
                              OLS Regression Results                             
Dep. Variable:     Mitotic_duration_mins   R-squared:                       0.081
Model:                               OLS   Adj. R-squared:                  0.077
Method:                    Least Squares   F-statistic:                     20.50
Date:                   Mon, 16 Mar 2026   Prob (F-statistic):           2.94e-09
Time:                           22:33:10   Log-Likelihood:                -2218.8
No. Observations:                    467   AIC:                             4444.
Df Residuals:                       

/var/folders/lr/cb1nd6l97696j8chvkss86580000gn/T/ipykernel_62661/2553710070.py:38: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  aov['omega_sq'] = (aov[:-1]['sum_sq']-(aov[:-1]['df'] * aov['mean_sq'][-1])) / (sum(aov['sum_sq']) + aov['mean_sq'][-1])




 POST-HOC testing for Mitotic_duration_mins: 

          Multiple Comparison of Means - Tukey HSD, FWER=0.05          
     group1          group2      meandiff p-adj   lower   upper  reject
-----------------------------------------------------------------------
       1_siLUC1  3_siLUC1siHAUS6  19.8802    0.0 11.4615 28.2989   True
       1_siLUC1 4_siUSP28siHAUS6  20.2122    0.0 12.2727 28.1517   True
3_siLUC1siHAUS6 4_siUSP28siHAUS6    0.332 0.9932 -6.6818  7.3458  False
-----------------------------------------------------------------------
If "reject" = True, then H0 should be rejected

 Welch's ANOVA table
      Source  ddof1       ddof2          F         p-unc       np2
0  Condition      2  287.891504  30.257296  1.183318e-12  0.081188

 Games-Howell post-hoc test table
                 A                 B    mean(A)    mean(B)       diff  \
0         1_siLUC1   3_siLUC1siHAUS6  65.446602  85.326797 -19.880195   
1         1_siLUC1  4_siUSP28siHAUS6  65.446602  85.658768 -20.